In [24]:
from calendar import weekday
import datetime as dt
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels as sm
import sklearn
import math
from statsmodels.formula.api import ols
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from datetime import date, timedelta
import holidays

In [25]:
def plotShit(errors):

    plt.figure()
    sns.histplot(errors)
    plt.figure()
    sns.lineplot(errors)
    plot_acf(errors)
    plot_pacf(errors)

    return

def isWorkday(date):
    hol = holidays.FI()
    if date.weekday() in (5,6):
        return False
    elif date in hol:
        return False
    else:
        return True


In [32]:





def fitModel(date,data, plotshit=True):

    df1 = data[data["Date"]<=date]



    df1 = df1[df1["l2"]!=0]

    m = ols(formula="log_consumption ~ C(month) + C(workday) + trend + l1_log", data=df1)
    r = m.fit()

    if plotshit:
        err = r.predict()-df1["log_consumption"]
        plotShit(err)
        ##
        plt.figure()
        sns.lineplot(df1["log_consumption"],color="r")
        sns.lineplot(r.predict())

    tplus1 = date+ timedelta(days=1)
    tplus2 = date+ timedelta(days=2)
    trend0 = max(df1["trend"])
    xt1 =  {"month" : tplus1.month,
           "workday" : isWorkday(tplus1),
           "trend" : trend0+1,
           "l1_log" : df1.iloc[-1,:]["l1_log"]}

    yt1 = r.predict(xt1)
    xt2 = {"month" : tplus2.month,
           "workday" : isWorkday(tplus2),
           "trend" : trend0+2,
           "l1_log" : yt1}
    yt2 = r.predict(xt2)

    return r, yt2




In [ ]:
#WEEK 1
df = pd.read_csv("Dailydf.csv",parse_dates=["Date"])


In [33]:
df1 = pd.read_csv("Dailydf.csv",parse_dates=["Date"])

df_spring = df1[df1["Date"]>=dt.datetime(2026,1,1)].set_index("Date").loc[:,["log_consumption","trend"]]
df_spring["predicted"] = pd.Series()

date = dt.datetime(2026,1,1)

while date<dt.datetime(2026,3,18):
    r,y = fitModel(date-dt.timedelta(days=2),df1, False)
    df_spring.loc[date, "predicted"] = y[0]
    date+=dt.timedelta(days=1)


